In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


In [2]:
!pip install wandb

In [3]:
!pip install -q faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 65.6 MB/s eta 0:00:00:00:0100:01


In [4]:
!pip install -q langchain

In [5]:
!pip install -q langchain-community

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 31.2 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 50.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 558.3/558.3 kB 34.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 5.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.39.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-adk 1.29.0 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
ydata-profiling 4.18.4 requires numba<0.63,>=0.60, but you have numba 0.65.1 which is incompatible.
ydata-profiling 4.18.4 requires numpy<2.4,>=1.22, but you have numpy 2.4.6 which is incompatible.
google-colab 1.0.0 requires jupyter-server==2.14.0, but you have jupyter-server 2.12.5 which is incompatible.
google-co

In [6]:
!pip install -q sentence-transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 92.1 MB/s eta 0:00:00:00:0100:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cudf-cu12 26.2.1 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 whic

In [7]:
!pip install transformers

## imports

In [ ]:
import wandb
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from torch.utils.data import TensorDataset,DataLoader
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
from sklearn.feature_extraction.text import TfidfVectorizer
import torch
import torch.nn as nn
from langchain_community.embeddings import HuggingFaceEmbeddings
import faiss
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document
from sentence_transformers import SentenceTransformer, util
from tqdm.auto import tqdm
from sklearn.metrics.pairwise import cosine_similarity
from collections import Counter
from datasets import load_dataset
from transformers import AutoTokenizer,AutoModel,pipeline,T5ForConditionalGeneration, T5Tokenizer,AutoModelForMultipleChoice
from peft import LoraConfig, get_peft_model, TaskType

/tmp/ipykernel_58/1554129085.py:9: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.embeddings import HuggingFaceEmbeddings


In [ ]:
# wandb.login()

In [ ]:
from kaggle_secrets import UserSecretsClient
import wandb
user_secrets = UserSecretsClient()
api_key = user_secrets.get_secret("WANDB_API_KEY")
wandb.login(key=api_key)

## wandb initilization and stuffs...

In [ ]:
run = wandb.init(
    entity="dlgenai-t22026",
    project="dlgenai-t226",
    name="model1-dl-genai",
    config={"model": "TF-IDF + Cosine Similarity",
    "dataset": "Smart MCQ Solver",
    "task": "MCQ Answer Ranking",
    "metric": "MAP@3"
    }
)

## loading data

In [ ]:
sample=pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv")

In [ ]:
train_df=pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")
test_df=pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv")

## EDA

In [ ]:
train_df.head()

In [ ]:
train_df.shape

### **the training data has 2000 rows and 8 columns**

In [ ]:
train_df.info()

In [ ]:
train_df.describe()

### most of them are text so cant get much insight from descriptive stats

In [ ]:
train_df.isnull().sum()

### **No missing values**

In [ ]:
train_df["answer"].value_counts()

In [ ]:
train_df["answer"].value_counts().sort_index().plot(kind="bar")
plt.title("Distribution of Answers")
plt.xlabel("Answer")
plt.ylabel("Count")
plt.show()

### **it has a little imbalance not much**

# DATA PREPROCESSING

## Data Split

In [ ]:
train_data, val_data = train_test_split(
    train_df,
    test_size=0.2,
    random_state=42,
    stratify=train_df["answer"])

### **we use stratify to keep the same answer distribution in both the training and validation datasets...for more reliable evaluation**

In [ ]:
print(f"Training Samples   : {len(train_data)}")
print(f"Validation Samples : {len(val_data)}")

# FEATURE ENGINEERING

In [ ]:
def create_samples(df):
    texts = []
    labels = []
    options = ["A", "B", "C", "D", "E"]
    for _, row in df.iterrows():
        prompt = row["prompt"]
        for option in options:
            text = prompt + " " + str(row[option])
            texts.append(text)
            if option == row["answer"]:
                labels.append(1)
            else:
                labels.append(0)
    return texts, labels

In [ ]:
train_texts, train_labels = create_samples(train_data)
val_texts, val_labels = create_samples(val_data)
print("Training Samples :", len(train_texts))
print("Validation Samples:", len(val_texts))

## Convert Text into TF-IDF Features

In [ ]:
vectorizer = TfidfVectorizer(
    max_features=10000,
    stop_words="english",
    ngram_range=(1, 2)
)

In [ ]:
X_train = vectorizer.fit_transform(train_texts)
X_val = vectorizer.transform(val_texts)

### **converted the text into numerical feature vectors**

### **pytorch works with tensors so we have to convert it first**

## PyTorch Data Pipeline

In [ ]:
X_train_tensor = torch.tensor(X_train.toarray(), dtype=torch.float32)
X_val_tensor = torch.tensor(X_val.toarray(), dtype=torch.float32)

In [ ]:
y_train_tensor = torch.tensor(train_labels, dtype=torch.float32)
y_val_tensor = torch.tensor(val_labels, dtype=torch.float32)

### **we have converted them to tensors ....these tensors will be used to train the neural network**

## TensorDataset

In [ ]:
train_dataset = TensorDataset(
    X_train_tensor,
    y_train_tensor
)

In [ ]:
val_dataset = TensorDataset(
    X_val_tensor,
    y_val_tensor
)

In [ ]:
BATCH_SIZE = 64
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True
)

In [ ]:
val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

In [ ]:
print("Training Batches :", len(train_loader))
print("Validation Batches:", len(val_loader))

# MODEL 1

## Build the Neural Network

In [ ]:
input_size = X_train_tensor.shape[1]
print("Input Size:", input_size)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using Device:", device)

In [ ]:
input_size = X_train_tensor.shape[1]

In [ ]:
class MCQClassifier(nn.Module):
    def __init__(self, input_size):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(input_size, 512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 1)
        )
    def forward(self, x):
        return self.network(x)

In [ ]:
model = MCQClassifier(input_size).to(device)
print(model)

In [ ]:
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)

# Model training

In [ ]:
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    for X_batch, y_batch in loader:
        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device).unsqueeze(1)
        optimizer.zero_grad()
        outputs = model(X_batch)
        loss = criterion(outputs, y_batch)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
    return running_loss / len(loader)

## validate

In [ ]:
def validate(model, loader, criterion, device):
    model.eval()
    running_loss = 0.0
    with torch.no_grad():
        for X_batch, y_batch in loader:
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device).unsqueeze(1)
            outputs = model(X_batch)
            loss = criterion(outputs, y_batch)
            running_loss += loss.item()
    return running_loss / len(loader)

## training loop

In [ ]:
best_val_loss = float("inf")
EPOCHS = 20
for epoch in range(EPOCHS):
    train_loss = train_one_epoch(
        model,
        train_loader,
        criterion,
        optimizer,
        device
    )
    val_loss = validate(
        model,
        val_loader,
        criterion,
        device
    )
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), "best_model.pth")
        print(f"✅ Best model saved! Validation Loss: {val_loss:.4f}")
    print(
        f"Epoch [{epoch+1}/{EPOCHS}] | "
        f"Train Loss: {train_loss:.4f} | "
        f"Validation Loss: {val_loss:.4f}"
    )
    wandb.log({
        "epoch": epoch + 1,
        "train_loss": train_loss,
        "validation_loss": val_loss
    })

## Model Evaluation

In [ ]:
model.eval()
predictions = []
with torch.no_grad():
    for X_batch, _ in val_loader:
        X_batch = X_batch.to(device)
        outputs = model(X_batch)
        scores = torch.sigmoid(outputs)
        predictions.extend(scores.squeeze().cpu().numpy())

In [ ]:
option_names = ["A", "B", "C", "D", "E"]
predicted_answers = []
for i in range(0, len(predictions), 5):
    scores = predictions[i:i+5]
    ranked = sorted(
        zip(option_names, scores),
        key=lambda x: x[1],
        reverse=True
    )
    top3 = [option for option, _ in ranked[:3]]
    predicted_answers.append(top3)

In [ ]:
true_answers = val_data["answer"].tolist()

In [ ]:
def apk(actual, predicted, k=3):
    if actual in predicted[:k]:
        return 1 / (predicted.index(actual) + 1)
    return 0
def mapk(actuals, predictions, k=3):
    scores = []
    for actual, predicted in zip(actuals, predictions):
        scores.append(apk(actual, predicted, k))
    return sum(scores) / len(scores)

In [ ]:
map3_score = mapk(
    true_answers,
    predicted_answers
)
print(f"Validation MAP@3: {map3_score:.4f}")
wandb.log({
    "MAP@3": map3_score
})

### i dont think 0.9671 is possible there must be some issue

In [ ]:
for i in range(5):
    print(f"Question {i+1}")
    print("Actual:", true_answers[i])
    print("Predicted:", predicted_answers[i])
    print()

In [ ]:
correct_top1 = 0
for actual, pred in zip(true_answers, predicted_answers):
    if actual == pred[0]:
        correct_top1 += 1
print(correct_top1)
print(len(true_answers))
print(correct_top1 / len(true_answers))

In [ ]:
model.load_state_dict(torch.load("best_model.pth"))
model.eval()
print("Model saved successfully!")

### **Tried multiple things but my scored seems to be stuck at 0.74854 maybe i should move on to the second model now....i tried with different learning rates but my scores started dropping slightly so i conclude this is what i can do with this architecture**

## Submit

In [ ]:
test_texts = []
for _, row in test_df.iterrows():
    prompt = row["prompt"]
    for option in ["A", "B", "C", "D", "E"]:
        text = prompt + " " + str(row[option])
        test_texts.append(text)

In [ ]:
X_test = vectorizer.transform(test_texts)

In [ ]:
X_test_tensor = torch.tensor(
    X_test.toarray(),
    dtype=torch.float32
)
test_dataset = TensorDataset(X_test_tensor)
test_loader = DataLoader(
    test_dataset,
    batch_size=64,
    shuffle=False
)

In [ ]:
model.eval()
test_scores = []
with torch.no_grad():
    for (X_batch,) in test_loader:
        X_batch = X_batch.to(device)
        outputs = model(X_batch)
        scores = torch.sigmoid(outputs)
        test_scores.extend(scores.squeeze().cpu().numpy())

In [ ]:
option_names = ["A", "B", "C", "D", "E"]
predictions = []
for i in range(0, len(test_scores), 5):
    scores = test_scores[i:i+5]
    ranked = sorted(
        zip(option_names, scores),
        key=lambda x: x[1],
        reverse=True
    )
    top3 = " ".join([option for option, _ in ranked[:3]])
    predictions.append(top3)

In [ ]:
submission = test_df[["id"]].copy()
submission["prediction"] = predictions
submission.head()

In [ ]:
submission.to_csv("submission_model1.csv", index=False)
print("Submission file saved successfully!")

# MODEL 2

## Vanilla RAG using LangChain + FAISS

In [ ]:
run = wandb.init(
    entity="dlgenai-t22026",
    project="dlgenai-t226",
    name="model2-langchain-rag",
    config={
        "model": "LangChain Vanilla RAG",
        "embedding_model": "all-MiniLM-L6-v2",
        "vector_database": "FAISS",
        "retriever_k": 5
    }
)

# MILESTONE 1

### Q1)

In [ ]:
# train_df["answer"].value_counts()

### Q2)

In [ ]:
# cleaned=train_df["prompt"].str.lower()

In [ ]:
# import string

In [ ]:
# for char in string.punctuation:
#     cleaned=cleaned.str.replace(char,"",regex=False)

In [ ]:
# len(cleaned.str.split().explode().unique())

### Q3)

In [ ]:
# words = cleaned.iloc[1].split()
# filtered = [word for word in words if word not in ENGLISH_STOP_WORDS]
# len(filtered)

### Q4)

In [ ]:
# combined = train_df[["prompt", "A", "B", "C", "D", "E"]].apply(lambda row: " ".join(row), axis=1)

In [ ]:
# vectorizer = TfidfVectorizer(stop_words="english")
# vectorizer.fit_transform(combined)

In [ ]:
# len(vectorizer.vocabulary_)

### Q5)

In [ ]:
# prompt_vec = vectorizer.transform([train_df.loc[1, "prompt"]])
# option_a_vec = vectorizer.transform([train_df.loc[1, "A"]])

In [ ]:
# score = cosine_similarity(prompt_vec, option_a_vec)
# round(float(score[0][0]), 4)

### Q6)

In [ ]:
# correct = 0
# for idx, row in train_df.iterrows():
#     prompt_vec = vectorizer.transform([row["prompt"]])
#     scores = {}
#     for option in ["A", "B", "C", "D", "E"]:
#         option_vec = vectorizer.transform([row[option]])
#         scores[option] = cosine_similarity(prompt_vec, option_vec)[0][0]
#     best_option = max(scores, key=lambda x: scores[x])
#     if best_option == row["answer"]:
#         correct += 1

In [ ]:
# percentage = round((correct / len(train_df)) * 100, 2)
# print(percentage)

### Q7)

In [ ]:
# top3 = train_df["answer"].value_counts().index[:3].tolist()

In [ ]:
# def apk(actual, predicted, k=3):
#     if actual in predicted[:k]:
#         return 1 / (predicted.index(actual) + 1)
#     return 0

In [ ]:
# scores = train_df["answer"].apply(lambda x: apk(x, top3))
# map3 = round(scores.mean(), 4)
# print(map3)

### Q8)

In [ ]:
# map3_scores = []

In [ ]:
# for idx, row in train_df.iterrows():
#     prompt_vec = vectorizer.transform([row["prompt"]])
#     sim_scores = {}
#     for option in ["A", "B", "C", "D", "E"]:
#         option_vec = vectorizer.transform([row[option]])
#         sim_scores[option] = cosine_similarity(prompt_vec, option_vec)[0][0]
#     top3_preds = sorted(sim_scores, key=lambda x: sim_scores[x], reverse=True)[:3]
#     map3_scores.append(apk(row["answer"], top3_preds))

In [ ]:
# print(round(sum(map3_scores) / len(map3_scores), 4))

# MILESTONE 2

### Q1)

In [ ]:
# dataset = load_dataset("csv", data_files="/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")["train"]
# def combine(row):
#     return {"combined_text": row["prompt"] + " " + row["A"]}
# dataset = dataset.map(combine)
# print(len(dataset[51]["combined_text"]))

### Q2)

In [ ]:
# tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
# print(tokenizer.vocab_size)

### Q3)

In [ ]:
# print(tokenizer.sep_token_id)

### Q4)

In [ ]:
# prompts = list(dataset["prompt"])
# tokens = tokenizer(prompts, padding='max_length', truncation=True, max_length=128, return_tensors='pt')
# print(tokens["input_ids"].shape)

### Q5)

In [ ]:
# model = AutoModel.from_pretrained("bert-base-uncased")
# prompt = dataset[0]["prompt"]
# tokens = tokenizer(prompt, return_tensors='pt')
# output = model(**tokens)
# print(output.last_hidden_state.shape)

### Q6)

In [ ]:
# cls_embedding = output.last_hidden_state[0, 0, :]
# print(round(cls_embedding[:5].sum().item(), 4))

### Q7)

In [ ]:
# model = AutoModel.from_pretrained("bert-base-uncased", output_attentions=True)
# text = "Light-ion fusion is a technique."
# tokens = tokenizer(text, return_tensors='pt')
# print(tokenizer.convert_ids_to_tokens(tokens['input_ids'][0]))
# output = model(**tokens)
# attentions = output.attentions[-1][0][0]
# token_list = tokenizer.convert_ids_to_tokens(tokens['input_ids'][0])
# fusion_idx = token_list.index('fusion')
# print(round(attentions[0][fusion_idx].item(), 4))

### Q8)

In [ ]:
# model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
# prompt = dataset[0]["prompt"]
# option_b = dataset[0]["B"]
# emb1 = model.encode(prompt)
# emb2 = model.encode(option_b)
# score = util.cos_sim(emb1, emb2)
# print(round(float(score), 4))

### Q9)

In [ ]:
# st_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

# options = ["A", "B", "C", "D", "E"]
# prompt_embs = st_model.encode(list(train_df["prompt"]), show_progress_bar=True)
# option_embs = {opt: st_model.encode(list(train_df[opt]), show_progress_bar=True) for opt in options}

# minilm_preds = []
# for i in range(len(train_df)):
#     scores = {}
#     for opt in options:
#         scores[opt] = float(util.cos_sim(prompt_embs[i], option_embs[opt][i]))
#     top3 = sorted(scores, key=lambda x: scores[x], reverse=True)[:3]
#     minilm_preds.append(top3)

# minilm_map3 = np.mean([apk(train_df.iloc[i]["answer"], minilm_preds[i]) for i in range(len(train_df))])
# print("MiniLM MAP@3:", round(minilm_map3, 4))

# train_df["combined"] = train_df[["prompt","A","B","C","D","E"]].apply(lambda row: " ".join(row), axis=1)
# vec = TfidfVectorizer(stop_words="english")
# tfidf_preds = []
# for i in range(len(train_df)):
#     prompt_vec = vec.fit_transform([train_df.iloc[i]["prompt"]])
#     sim_scores = {}
#     for opt in options:
#         opt_vec = vec.transform([train_df.iloc[i][opt]])
#         sim_scores[opt] = cosine_similarity(prompt_vec, opt_vec)[0][0]
#     top3 = sorted(sim_scores, key=lambda x: sim_scores[x], reverse=True)[:3]
#     tfidf_preds.append(top3)
# count = 0
# for i in range(len(train_df)):
#     answer = train_df.iloc[i]["answer"]
#     if answer not in tfidf_preds[i] and answer in minilm_preds[i]:
#         count += 1
# print("MiniLM wins over TF-IDF count:", count)

### Q10)

In [ ]:
# classifier = pipeline("zero-shot-classification")
# prompt = dataset[1]["prompt"]
# candidate_labels = [dataset[1]["A"], dataset[1]["B"], dataset[1]["C"]]
# result = classifier(prompt, candidate_labels)
# print(round(result["scores"][0], 4))

### Q11)

In [ ]:
# result_single = classifier(prompt, candidate_labels)
# result_multi = classifier(prompt, candidate_labels, multi_label=True)
# sum_single = sum(result_single["scores"])
# sum_multi = sum(result_multi["scores"])
# print(round(abs(sum_single - sum_multi), 4))

### Q12)

In [ ]:
# tokenizer_flan = T5Tokenizer.from_pretrained("google/flan-t5-small")
# model_flan = T5ForConditionalGeneration.from_pretrained("google/flan-t5-small")
# prompt_text = dataset[0]["prompt"]
# option_a = dataset[0]["A"]
# option_b = dataset[0]["B"]
# input_text = f"Question: {prompt_text}. Is the correct answer A: {option_a} or B: {option_b}? Answer with just the letter A or B."
# inputs = tokenizer_flan(input_text, return_tensors="pt")
# outputs = model_flan.generate(**inputs, max_new_tokens=5)
# print(tokenizer_flan.decode(outputs[0], skip_special_tokens=True))

# MILESTONE 3

### Q1)

In [ ]:
# !pip install -q faiss-cpu

# import pandas as pd
# import numpy as np
# import faiss
# from sentence_transformers import SentenceTransformer, CrossEncoder
# from transformers import AutoTokenizer, pipeline
# from sklearn.feature_extraction.text import TfidfVectorizer
# from sklearn.metrics.pairwise import cosine_similarity

# print("Creating knowledge base")
# kb = []
# for idx, row in train_df.iterrows():
#     correct_letter = row["answer"]
#     kb.append(str(row[correct_letter]))

# print("Loading embedding model and creating index")
# model = SentenceTransformer("all-MiniLM-L6-v2")
# kb_embeddings = model.encode(kb, show_progress_bar=False)

# index = faiss.IndexFlatL2(kb_embeddings.shape[1])
# index.add(kb_embeddings)

# print("Knowledge base successfully created")

# # Zero-shot classifier for Q1, Q2, Q6
# zs = pipeline(
#     "zero-shot-classification",
#     model="facebook/bart-large-mnli"
# )

# row_150 = train_df.iloc[150]
# prompt_150 = str(row_150["prompt"])
# labels_150 = [
#     str(row_150["A"]),
#     str(row_150["B"]),
#     str(row_150["C"]),
#     str(row_150["D"]),
#     str(row_150["E"])
# ]
# ans_150 = str(row_150[row_150["answer"]])

In [ ]:
# result = zs(prompt_150, candidate_labels=labels_150)
# print(result)

In [ ]:
# ground_truth_score = result["scores"][result["labels"].index(ans_150)]
# print(round(ground_truth_score, 3))

### Q2)

In [ ]:
# query_embedding = model.encode([prompt_150], show_progress_bar=False)
# D, I = index.search(query_embedding, k=10)
# print("Retrieved KB indices:")
# print(I[0])
# print("\nRetrieved documents:")
# for rank, idx in enumerate(I[0], start=1):
#     print(f"Rank {rank}: KB index {idx}")
#     print(kb[idx][:120], "...\n")
# if 150 in I[0]:
#     rank = list(I[0]).index(150) + 1
#     print(f"True document found at Rank {rank}")
# else:
#     print("True document is NOT in the top 10.")

### Q3)

In [ ]:
# cross_encoder = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')
# retrieved_indices = I[0]
# docs_10 = [kb[i] for i in retrieved_indices]
# pairs = [[prompt_150, doc] for doc in docs_10]
# ce_scores = cross_encoder.predict(pairs)

In [ ]:
# ranking = sorted(
#     zip(retrieved_indices, ce_scores),
#     key=lambda x: x[1],
#     reverse=True
# )
# for rank, (idx, score) in enumerate(ranking, start=1):
#     print(f"Rank {rank}: KB index {idx}, Score: {score:.4f}")
# for rank, (idx, score) in enumerate(ranking, start=1):
#     if idx == 150:
#         print(f"\nTrue document found at Rank {rank}")
#         break

### Q4)

In [ ]:
# row_42 = train_df.iloc[42]
# prompt_42 = str(row_42["prompt"])
# query_embedding = model.encode([prompt_42], show_progress_bar=False)
# D, I = index.search(query_embedding, k=5)
# docs_5 = [kb[i] for i in I[0]]
# concatenated_docs = " ".join(docs_5)
# text = f"Context: {concatenated_docs} Question: {prompt_42}"
# tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
# tokens = tokenizer(text, truncation=False)
# print("Total tokens:", len(tokens["input_ids"]))

### Q5)

In [ ]:
# row_150 = train_df.iloc[150]
# prompt_150 = str(row_150["prompt"])
# true_document = kb[150]
# rag_prompt = f"Context: {true_document} Question: {prompt_150}"
# result_rag = zs(
#     rag_prompt,
#     candidate_labels=labels_150
# )
# print(result_rag)
# ground_truth_score = result_rag["scores"][result_rag["labels"].index(ans_150)]
# print("Ground-truth score:", round(ground_truth_score, 3))

### Q6)

In [ ]:
# row_150 = train_df.iloc[150]
# prompt_150 = str(row_150["prompt"])
# wrong_document = kb[999]
# adversarial_prompt = f"Context: {wrong_document} Question: {prompt_150}"
# result_adv = zs(
#     adversarial_prompt,
#     candidate_labels=labels_150
# )
# print(result_adv)
# ground_truth_score = result_adv["scores"][result_adv["labels"].index(ans_150)]
# print("Ground-truth score:", round(ground_truth_score, 3))

### Q7)

In [ ]:
# hits = 0
# for i in range(100):
#     prompt = str(train_df.iloc[i]["prompt"])
#     correct_option = train_df.iloc[i][train_df.iloc[i]["answer"]]
#     query_embedding = model.encode([prompt], show_progress_bar=False)
#     D, I = index.search(query_embedding, k=5)
#     retrieved_docs = [kb[idx] for idx in I[0]]
#     if any(correct_option == doc for doc in retrieved_docs):
#         hits += 1
# hit_rate = (hits / 100) * 100
# print("Hits:", hits)
# print("Hit Rate:", round(hit_rate, 1))

### Q8)

In [ ]:
# def apk(actual, predicted, k=3):
#     if actual in predicted[:k]:
#         return 1 / (predicted.index(actual) + 1)
#     return 0
# scores = []
# for i in range(20):
#     row = train_df.iloc[i]
#     prompt = str(row["prompt"])
#     query_embedding = model.encode([prompt], show_progress_bar=False)
#     D, I = index.search(query_embedding, k=5)
#     retrieved_indices = I[0]
#     docs = [kb[idx] for idx in retrieved_indices]
#     pairs = [[prompt, doc] for doc in docs]
#     ce_scores = cross_encoder.predict(pairs)
#     best_doc = docs[np.argmax(ce_scores)]
#     rag_prompt = f"Context: {best_doc} Question: {prompt}"
#     labels = [
#         str(row["A"]),
#         str(row["B"]),
#         str(row["C"]),
#         str(row["D"]),
#         str(row["E"])
#     ]
#     result = zs(rag_prompt, candidate_labels=labels)
#     option_map = {
#         row["A"]: "A",
#         row["B"]: "B",
#         row["C"]: "C",
#         row["D"]: "D",
#         row["E"]: "E"
#     }
#     predicted_letters = [option_map[label] for label in result["labels"]]
#     scores.append(apk(row["answer"], predicted_letters))
# final_map3 = np.mean(scores)
# print("MAP@3:", round(final_map3, 3))

# MILESTONE 4

In [ ]:
# label_map = {
#     "A": 0,
#     "B": 1,
#     "C": 2,
#     "D": 3,
#     "E": 4
# }

In [ ]:
# train_df["label"] = train_df["answer"].map(label_map)
# print(train_df.loc[150, "label"])

In [ ]:
# formatted = str(train_df.loc[0, "prompt"]) + " [SEP] " + str(train_df.loc[0, "B"])
# len(formatted)

In [ ]:
# tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
# row = train_df.loc[0]

# choices = [
#     str(row["prompt"]) + " [SEP] " + str(row["A"]),
#     str(row["prompt"]) + " [SEP] " + str(row["B"]),
#     str(row["prompt"]) + " [SEP] " + str(row["C"]),
#     str(row["prompt"]) + " [SEP] " + str(row["D"]),
#     str(row["prompt"]) + " [SEP] " + str(row["E"]),
# ]

# enc = tokenizer(
#     choices,
#     padding="max_length",
#     truncation=True,
#     max_length=128,
#     return_tensors="pt"
# )

# input_ids = enc["input_ids"].unsqueeze(0)
# print(input_ids.shape)

In [ ]:
# input_ids.shape

In [ ]:
# input_ids.numel()

In [ ]:
# model = AutoModelForMultipleChoice.from_pretrained("bert-base-uncased")

In [ ]:
# lora_config = LoraConfig(
#     r=8,
#     lora_alpha=16,
#     target_modules=["query", "value"],
#     lora_dropout=0.1,
#     bias="none",
#     task_type=TaskType.SEQ_CLS
# )

In [ ]:
# model = get_peft_model(model, lora_config)

In [ ]:
# trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
# print(trainable_params)

In [ ]:
# row = train_df.loc[0]
# choices = [
#     f"{row['prompt']} [SEP] {row['A']}",
#     f"{row['prompt']} [SEP] {row['B']}",
#     f"{row['prompt']} [SEP] {row['C']}",
#     f"{row['prompt']} [SEP] {row['D']}",
#     f"{row['prompt']} [SEP] {row['E']}",
# ]
# enc = tokenizer(
#     choices,
#     padding="max_length",
#     truncation=True,
#     max_length=64,      # same as Q9
#     return_tensors="pt"
# )
# input_ids = enc["input_ids"].unsqueeze(0)
# attention_mask = enc["attention_mask"].unsqueeze(0)

In [ ]:
# model.eval()
# with torch.no_grad():
#     outputs = model(
#         input_ids=input_ids,
#         attention_mask=attention_mask
#     )

In [ ]:
# probs = torch.softmax(outputs.logits, dim=1)
# print(probs)